Логистическая регрессия может быть обобщена к поддержке мультиклассов,
без необходимости обучать и объединять всё отдельно.
Вместо нескольких логистических регрессий:

- кошка vs собака
- кошка vs крыса
- собака vs крыса

можно использовать одну Softmax Regression, которая сразу считает вероятность для всех трёх:

- кошка → 70%
- собака → 20%
- крыса → 10%

То есть Softmax — это способ расширить логистическую регрессию с бинарной классификации на многоклассовую.

> Бинарный = есть только два возможных класса/варианта.

---

## Score

Формула:

$$
s_k(x) = \boldsymbol{\theta}^{(k)\top} x
$$

### Два признака

- $x_1$ = размер
- $x_2$ = вес

### Три класса

- $k = 1$ → cat
- $k = 2$ → dog
- $k = 3$ → rabbit

### У каждого класса свои параметры

$$
\theta^{(1)} = [2, 1]
$$

$$
\theta^{(2)} = [1, 2]
$$

$$
\theta^{(3)} = [0.5, 1]
$$

А объект:

$$
x = [3, 2]
$$

### 1. Score подсчёт

**Cat:**

$$
s_1 = 2 \cdot 3 + 1 \cdot 2 = 8
$$

**Dog:**

$$
s_2 = 1 \cdot 3 + 2 \cdot 2 = 7
$$

**Rabbit:**

$$
s_3 = 0.5 \cdot 3 + 1 \cdot 2 = 3.5
$$

Это пока не вероятности, а оценки того, насколько объект подходит каждому классу.

---

## 2. Превращение в вероятности

$$
\hat{p}_k = \frac{e^{s_k}}{\sum_j e^{s_j}}
$$

Softmax говорит: есть оценки `8`, `7`, `3.5` — давайте превратим их в вероятности, сумма которых будет 100%.

### 1. Подсчёт экспоненты каждого Score

- Cat: $e^8 \approx 2980.96$
- Dog: $e^7 \approx 1096.63$
- Rabbit: $e^{3.5} \approx 33.12$

### 2. Складываем их

$$
2980.96 + 1096.63 + 33.12 = 4110.71
$$

### 3. Считаем вероятность каждого класса

**Cat:**

$$
\hat{p}_1 =
\frac{e^8}{4110.71}
=
\frac{2980.96}{4110.71}
\approx 0.725
$$

**Dog:**

$$
\hat{p}_2 =
\frac{e^7}{4110.71}
=
\frac{1096.63}{4110.71}
\approx 0.267
$$

**Rabbit:**

$$
\hat{p}_3 =
\frac{e^{3.5}}{4110.71}
=
\frac{33.12}{4110.71}
\approx 0.008
$$

---

Softmax делает:

**Берёт scores → применяет $e$ → делит каждый результат на сумму всех → получает вероятности, сумма которых равна 1.**

---
## 3. Argmax

Верни индекс того элемента, у которого значение максимальное.

$$
\hat{y} = \arg\max_k s_k(x)
$$

Например:

$$
[8,\ 7,\ 3.5]
$$

Тогда:

$$
\arg\max [8,\ 7,\ 3.5] = 1
$$

То есть:

```text
Кот      → 8
Собака   → 7
Кролик   → 3.5

argmax → Кот
```


### 4. Cross Entropy — функция стоимости

При обучении мы хотим, чтобы модель давала **высокую вероятность правильному классу** и низкую — остальным.

**Cross entropy** измеряет, насколько хорошо предсказанные вероятности совпадают с правильным классом:

> чем ниже вероятность правильного класса, тем больше штраф.

Поэтому обучение старается **минимизировать Cross Entropy**.

---

## Коротко

$$
J(\Theta)
=
-\frac{1}{m}
\sum_{i=1}^{m}
\sum_{k=1}^{K}
y_k^{(i)}
\log\left(\hat{p}_k^{(i)}\right)
$$

Можно понимать так:

> **Для каждого объекта берём вероятность правильного класса, превращаем её в штраф через $-\log$, затем усредняем этот штраф по всем объектам.**

```python
loss = -log(target_probability)
```

Главная идея:

```text
высокая вероятность правильного класса
        ↓
     маленький loss

низкая вероятность правильного класса
        ↓
     большой loss
```

### Пример

Модель предсказала:

```text
Cat    → 0.725
Dog    → 0.267
Rabbit → 0.008
```

Правильный класс — **Cat**.

Тогда:

$$
L=-\log(0.725)\approx0.322
$$

Loss небольшой, потому что модель дала правильному классу высокую вероятность.

Если модель ошиблась и дала:

```text
Cat    → 0.01
Dog    → 0.80
Rabbit → 0.19
```

то:

$$
L=-\log(0.01)\approx4.605
$$

Loss становится намного больше.

Поэтому:

```text
P(correct) = 0.9  → loss ≈ 0.105
P(correct) = 0.5  → loss ≈ 0.693
P(correct) = 0.1  → loss ≈ 2.303
P(correct) = 0.01 → loss ≈ 4.605
```

**Чем сильнее модель уверена в неправильном ответе, тем сильнее её штрафуют.**

---

## Вся идея

```text
Score
  ↓
Softmax
  ↓
вероятности классов
  ↓
Cross Entropy
  ↓
loss
  ↓
Gradient
  ↓
обновление параметров
```

> **Softmax отвечает:** «Какие вероятности у классов?»

> **Cross Entropy отвечает:** «Насколько плохо распределены эти вероятности относительно правильного ответа?»

### 5. Gradient of Cross Entropy

После Cross Entropy мы знаем, **насколько хорошо модель предсказывает**.

Теперь нужно понять:

> **Как изменить параметры модели, чтобы loss стал меньше?**

Для этого используется **Gradient**.

---

## Коротко

Формула из книги:

$$
\nabla_{\boldsymbol{\theta}^{(k)}} J(\Theta)
=
\frac{1}{m}
\sum_{i=1}^{m}
\left(
\hat{p}_k^{(i)} - y_k^{(i)}
\right)
x^{(i)}
$$

Можно понимать так:

> **Gradient показывает, как параметры класса $k$ влияют на loss и в каком направлении loss растёт.**

Поэтому при обучении мы идём **в противоположную сторону** — используем Gradient Descent.

```python
gradient = mean(
    (prediction - target) * features
)
```

---

## Самая важная часть

$$
\hat{p}_k^{(i)} - y_k^{(i)}
$$

То есть:

```text
prediction - target
```

Например:

```text
Правильный класс = Cat

Cat    → 0.725 - 1 = -0.275
Dog    → 0.267 - 0 =  0.267
Rabbit → 0.008 - 0 =  0.008
```

Для правильного класса получается отрицательное значение, а для неправильных — положительное.

---

## Добавляем признаки $x$

$$
\left(
\hat{p}_k^{(i)} - y_k^{(i)}
\right)x^{(i)}
$$

Gradient зависит от:

```text
ошибка
  ×
признаки объекта
```

После этого вклад всех объектов усредняется.

---
## Gradient Descent

| Символ | Читается | Значение |
|---|---|---|
| θ | тета | параметры модели |
| η | эта | learning rate |
| ∇ | набла | значок «взять градиент» |
| Θ | тета (заглавная) | то же, что θ |

Чтобы уменьшить loss:

$$
\boldsymbol{\theta}
\leftarrow
\boldsymbol{\theta}
-
\eta
\nabla_{\boldsymbol{\theta}}J(\Theta)
$$

```python
theta = theta - learning_rate * gradient
```

Важно:

> **Gradient указывает направление роста loss, поэтому Gradient Descent двигается в противоположную сторону.**

---
---
## Вся цепочка

```text
x
↓
Score
↓
Softmax
↓
вероятности
↓
Cross Entropy
↓
loss
↓
Gradient
↓
Gradient Descent
↓
обновление параметров
```

### Главное

> **Cross Entropy говорит, насколько плохо модель предсказала.**

> **Gradient показывает, как изменение параметров влияет на этот loss.**

> **Gradient Descent использует Gradient, чтобы изменить параметры и уменьшить loss.**
